# Õppetund 04 - Tööriistade kasutamise disainimuster

Selles õppetükis õpite Microsoft Agent Frameworki (Python) kasutades AI agentide jaoks **tööriistade kasutamise** disainimustrit. Katame:

- Funktsiooni tööriistade defineerimise `@tool` dekoraatori ja tüüpidega parameetritega
- Tööriistade skeemide pakkumise, et mudel teaks, mida iga tööriist teeb
- Tööriistade täitmise juhtimise `approval_mode` abil
- **Struktureeritud väljundi** tagastamise Pydantic mudelite ja `response_format` kaudu

Stsenaariumiks on **reisibroneerimisagent**, kes saab otsida sihtkohti, kontrollida saadavust ja hankida lennuinfot.


## Seadistamine


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tööriistade defineerimine @tool dekoratsiooniga

`@tool` dekoratsioon muudab tavalise Pythoni funktsiooni tööriistaks, mida agent saab kutsuda.
Olulised punktid:

- **docstring** muutub tööriista kirjelduseks, mida mudel näeb.
- **Tüübiannotatsioonid** (sealhulgas `Annotated` koos kirjetega) määravad tööriista skeemi.
- `approval_mode` kontrollib, kas kasutaja peab iga kutse enne täitmist heaks kiitma.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Agendi loomine mitme tööriistaga

Edastage kõik kolm tööriista kliendile, et mudel saaks kasutada kõiki vajalikke tööriistu kasutaja küsimusele vastamiseks.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Struktureeritud väljund tööriistadega

Määrates `response_format` Pydantic mudelile, sunnitakse agent tagastama hästi tüübistatud JSON-objekti vaba vormi teksti asemel. See on kasulik, kui järgnevat koodi on vaja tulemust programmiliselt tarbida.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tööriista heakskiidu mustrid

Parameeter `approval_mode` @tool määrab, kas tööriista kutsed vajavad täitmiseks inimeste heakskiitu:

| Režiim | Käitumine |
|---|---|
| `"never_require"` | Tööriist töötab automaatselt — kasutaja kinnitust ei nõuta. |
| `"always_require"` | Iga kutse peab saama kasutaja heakskiidu enne täitmist. |

Kasuta `"always_require"` tööriistade puhul, mis põhjustavad kõrvalmõjusid (nt lennu broneerimine, krediitkaardi laadimine), et inimene oleks protsessis kaasatud.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Kokkuvõte

Selles õppetükis õppisid, kuidas:

1. **Määratleda tööriistu** kasutades `@tool` dekoratsiooni tüübitud parameetrite ja dokstringidega, mis toimivad tööriista skeemina.
2. **Koostada mitut tööriista** nii, et agent saab neid järjestikku kutsuda keeruliste päringute vastamiseks.
3. **Tagastada struktureeritud väljundit** edastades Pydantic mudeli kui `response_format`.
4. **Kontrollida tööriista heakskiitu** kasutades `approval_mode`, et hoida tundlike toimingute puhul inimene protsessis kaasatud.

Need mustrid moodustavad usaldusväärsete, tootmiskõlblike agentide loomise aluse, kes saavad turvaliselt suhelda väliste süsteemidega.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades tehisintellektil põhinevat tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüame tagada täpsust, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument oma emakeeles tuleks pidada ametlikuks allikaks. Olulise info puhul on soovitatav kasutada professionaalset inimtõlget. Me ei vastuta käesoleva tõlke kasutamisest tingitud arusaamatuste või valesti mõistmiste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
